# ADBE Owner Economics

The first notebook proved one narrow SEC/XBRL lesson: raw observations become a trustworthy canonical revenue series. This notebook uses the business-economics layer built on top of that foundation to answer the questions an owner actually asks.

Every number here is derived deterministically from Adobe's raw SEC Company Facts. Reported facts keep full provenance; margins, cash flow, per-share economics, and capital-efficiency ratios are calculated in OwnerLens, never sourced from an external field.

We answer eight questions:

1. How much is Adobe growing?
2. How profitable is that growth?
3. How much cash does the business actually generate?
4. How much cash generation belongs to each share?
5. Is the share count helping or hurting owners?
6. How strong is the balance sheet?
7. How efficiently is Adobe using capital?
8. What changed economically from FY2021 to FY2025?

> Set an SEC User-Agent identifying your application and an administrative contact. Replace the sample contact below before making live requests.

In [ ]:
import os

from owner_lens import (
    SecClient,
    capital_efficiency_from_facts,
    owner_economics_from_facts,
)

USER_AGENT = os.environ.get("OWNER_LENS_SEC_USER_AGENT", "OwnerLens admin@example.com")

raw = SecClient(USER_AGENT).retrieve_company_facts("ADBE").raw_facts
oe = {row.fiscal_year: row for row in owner_economics_from_facts(raw)}
ce = {row.fiscal_year: row for row in capital_efficiency_from_facts(raw)}
years = sorted(oe)


def money(value):
    return f"{value:,}" if value is not None else "n/a"


def billions(value):
    return f"{value / 1e9:,.2f}B" if value is not None else "n/a"


def pct(value):
    return f"{value:.1%}" if value is not None else "n/a"


def num(value, digits=2):
    return f"{value:.{digits}f}" if value is not None else "n/a"


def val(observation):
    return observation.value if observation is not None else None


def table(headers, rows):
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows)) for i, h in enumerate(headers)]
    line = "  ".join(str(h).rjust(widths[i]) for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(str(c).rjust(widths[i]) for i, c in enumerate(r)))


print("fiscal years:", years)

## 1. How much is Adobe growing?

Revenue is the top of the funnel. Year-over-year growth shows whether the business is still expanding and how fast.

In [ ]:
rows = []
for y in years:
    revenue = val(oe[y].revenue)
    prev = val(oe[y - 1].revenue) if (y - 1) in oe else None
    growth = (revenue / prev - 1) if revenue and prev else None
    rows.append([y, billions(revenue), pct(growth)])
table(["FY", "revenue", "YoY growth"], rows)

## 2. How profitable is that growth?

Growth only matters if it converts to profit. Operating margin reflects the core business; net margin reflects what reaches the bottom line after interest and taxes.

In [ ]:
rows = []
for y in years:
    r = oe[y]
    rows.append(
        [
            y,
            billions(val(r.operating_income)),
            pct(r.operating_margin),
            billions(val(r.net_income)),
            pct(r.net_margin),
        ]
    )
table(["FY", "op income", "op margin", "net income", "net margin"], rows)

## 3. How much cash does the business actually generate?

Accounting profit is not cash. Free cash flow is operating cash flow minus the capital the business must reinvest, and FCF margin shows how much of every revenue dollar becomes owner cash.

In [ ]:
rows = []
for y in years:
    r = oe[y]
    rows.append(
        [
            y,
            billions(val(r.operating_cash_flow)),
            billions(val(r.capital_expenditures)),
            billions(r.free_cash_flow),
            pct(r.fcf_margin),
        ]
    )
table(["FY", "OCF", "capex", "FCF", "FCF margin"], rows)

## 4. How much cash generation belongs to each share?

Owners hold shares, not the whole company. Free cash flow per diluted share translates total cash generation into the per-share economics an owner actually receives a claim on.

In [ ]:
rows = []
for y in years:
    r = oe[y]
    shares = val(r.diluted_shares)
    rows.append(
        [
            y,
            billions(r.free_cash_flow),
            f"{shares / 1e6:,.1f}M" if shares else "n/a",
            num(r.fcf_per_share),
        ]
    )
table(["FY", "FCF", "diluted shares", "FCF/share"], rows)

## 5. Is the share count helping or hurting owners?

Buybacks shrink the share count, so each remaining share claims more of the same cash. Comparing FCF growth to FCF-per-share growth shows how much of the per-share gain comes from the business versus from a falling share count.

In [ ]:
rows = []
for y in years:
    r = oe[y]
    rows.append(
        [
            y,
            pct(r.diluted_share_growth),
            pct(r.fcf_growth),
            pct(r.fcf_per_share_growth),
        ]
    )
table(["FY", "share count gr", "FCF growth", "FCF/share growth"], rows)

## 6. How strong is the balance sheet?

A strong balance sheet gives an owner staying power and optionality. Net cash is the liquid cushion after debt; assets and equity show the scale of capital the business stands on.

In [ ]:
rows = []
for y in years:
    r = ce[y]
    rows.append(
        [
            y,
            billions(r.cash_plus_sti),
            billions(r.total_debt),
            billions(r.net_cash),
            billions(val(r.total_assets)),
            billions(val(r.total_equity)),
        ]
    )
table(["FY", "cash+STI", "debt", "net cash", "assets", "equity"], rows)

## 7. How efficiently is Adobe using capital?

Returns on capital reveal whether growth creates value. ROA and ROE measure accounting returns; ROIC compares after-tax operating profit (NOPAT) to the capital actually invested in the business, using average balances.

In [ ]:
rows = []
for y in years:
    r = ce[y]
    rows.append(
        [
            y,
            billions(r.nopat),
            billions(r.invested_capital),
            pct(r.roa),
            pct(r.roe),
            pct(r.roic),
        ]
    )
table(["FY", "NOPAT", "invested cap", "ROA", "ROE", "ROIC"], rows)

## 8. What changed economically from FY2021 to FY2025?

Putting the endpoints side by side summarizes the owner's economic story over the window: bigger revenue, more cash per share, and higher returns on capital.

In [ ]:
first, last = years[0], years[-1]


def change(a, b):
    return (b / a - 1) if a and b else None


metrics = [
    ("revenue", billions(val(oe[first].revenue)), billions(val(oe[last].revenue)), change(val(oe[first].revenue), val(oe[last].revenue))),
    ("FCF", billions(oe[first].free_cash_flow), billions(oe[last].free_cash_flow), change(oe[first].free_cash_flow, oe[last].free_cash_flow)),
    ("FCF/share", num(oe[first].fcf_per_share), num(oe[last].fcf_per_share), change(oe[first].fcf_per_share, oe[last].fcf_per_share)),
    ("diluted shares", f"{val(oe[first].diluted_shares) / 1e6:,.1f}M", f"{val(oe[last].diluted_shares) / 1e6:,.1f}M", change(val(oe[first].diluted_shares), val(oe[last].diluted_shares))),
    ("net cash", billions(ce[first].net_cash), billions(ce[last].net_cash), change(ce[first].net_cash, ce[last].net_cash)),
    ("ROIC", pct(ce[first].roic), pct(ce[last].roic), None),
]
table(
    ["metric", f"FY{first}", f"FY{last}", "change"],
    [[name, a, b, pct(delta)] for name, a, b, delta in metrics],
)

## Takeaway

> Owner economics is a chain, not a single number: growth must convert to margin, margin to cash, cash to per-share cash, and capital to returns. Reading these questions together, rather than any one metric alone, is how OwnerLens judges whether a business is compounding value for its owners.